In [6]:
"""
================================================================================
STEP 2: BUILD kNN GRAPH FROM BERT EMBEDDINGS
================================================================================
Creates a k-nearest neighbors graph using cosine similarity.
Includes comprehensive validation to ensure everything is correct.
================================================================================
"""

import numpy as np
import scipy.sparse as sp
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import json
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("BUILDING kNN GRAPH FOR GCN")
print("="*80)

BUILDING kNN GRAPH FOR GCN


In [7]:
# ============================================================================
# CONFIGURATION
# ============================================================================
class GraphConfig:
    # Input files
    EMBEDDINGS_PATH = 'node_features.npy'
    LABELS_PATH = 'labels.npy'
    TRAIN_MASK_PATH = 'train_mask.npy'
    VAL_MASK_PATH = 'val_mask.npy'
    TEST_MASK_PATH = 'test_mask.npy'
    
    # Graph parameters
    K_NEIGHBORS = 10  # Number of nearest neighbors
    USE_BINARY_EDGES = True  # Binary (1/0) vs weighted (similarity values)
    MAKE_SYMMETRIC = True  # Symmetrize the graph
    
    # Output files
    ADJ_OUTPUT_PATH = 'adjacency_matrix.npz'
    EDGE_LIST_OUTPUT_PATH = 'edge_list.txt'
    GRAPH_STATS_PATH = 'graph_statistics.json'

config = GraphConfig()

In [8]:
# ============================================================================
# STEP 1: LOAD DATA
# ============================================================================
print("\n" + "="*80)
print("STEP 1: LOAD DATA")
print("="*80)

print("Loading embeddings...")
embeddings = np.load(config.EMBEDDINGS_PATH)
print(f"✓ Embeddings shape: {embeddings.shape}")

print("\nLoading labels...")
labels = np.load(config.LABELS_PATH)
print(f"✓ Labels shape: {labels.shape}")

print("\nLoading masks...")
train_mask = np.load(config.TRAIN_MASK_PATH)
val_mask = np.load(config.VAL_MASK_PATH)
test_mask = np.load(config.TEST_MASK_PATH)
print(f"✓ Train mask: {train_mask.sum():,} nodes")
print(f"✓ Val mask: {val_mask.sum():,} nodes")
print(f"✓ Test mask: {test_mask.sum():,} nodes")


STEP 1: LOAD DATA
Loading embeddings...
✓ Embeddings shape: (122817, 768)

Loading labels...
✓ Labels shape: (122817,)

Loading masks...
✓ Train mask: 12,281 nodes
✓ Val mask: 6,141 nodes
✓ Test mask: 104,395 nodes


In [9]:
# ============================================================================
# STEP 2: VALIDATE DATA
# ============================================================================
print("\n" + "="*80)
print("STEP 2: VALIDATE DATA")
print("="*80)

n_nodes = embeddings.shape[0]
print(f"Total nodes: {n_nodes:,}")

# Check 1: Shapes match
assert embeddings.shape[0] == labels.shape[0], "Embeddings and labels size mismatch!"
assert embeddings.shape[0] == train_mask.shape[0], "Embeddings and train_mask size mismatch!"
assert embeddings.shape[0] == val_mask.shape[0], "Embeddings and val_mask size mismatch!"
assert embeddings.shape[0] == test_mask.shape[0], "Embeddings and test_mask size mismatch!"
print("✓ All shapes match")

# Check 2: No overlap in masks
assert not np.any(train_mask & val_mask), "Train/Val overlap detected!"
assert not np.any(train_mask & test_mask), "Train/Test overlap detected!"
assert not np.any(val_mask & test_mask), "Val/Test overlap detected!"
print("✓ No mask overlap")

# Check 3: Masks cover all nodes
total_masked = (train_mask | val_mask | test_mask).sum()
assert total_masked == n_nodes, f"Masks don't cover all nodes! {total_masked} vs {n_nodes}"
print("✓ Masks cover all nodes")

# Check 4: No NaN/Inf in embeddings
assert not np.isnan(embeddings).any(), "NaN values in embeddings!"
assert not np.isinf(embeddings).any(), "Inf values in embeddings!"
print("✓ No NaN/Inf in embeddings")

print("\n✅ All validation checks passed!")


STEP 2: VALIDATE DATA
Total nodes: 122,817
✓ All shapes match
✓ No mask overlap
✓ Masks cover all nodes
✓ No NaN/Inf in embeddings

✅ All validation checks passed!


In [10]:
# ============================================================================
# STEP 3: BUILD kNN GRAPH (EFFICIENT - NO FULL SIMILARITY MATRIX)
# ============================================================================
print("\n" + "="*80)
print("STEP 3: BUILD kNN GRAPH (EFFICIENT METHOD)")
print("="*80)

print(f"Building graph with k={config.K_NEIGHBORS} nearest neighbors...")
print("Using sklearn NearestNeighbors (efficient for 122k documents)")

from sklearn.neighbors import NearestNeighbors

# Use NearestNeighbors with cosine similarity
# This is MUCH more memory efficient than full pairwise similarity
nbrs = NearestNeighbors(
    n_neighbors=config.K_NEIGHBORS + 1,  # +1 because includes self
    metric='cosine',
    algorithm='brute',  # Exact for accuracy
    n_jobs=-1  # Use all CPU cores
).fit(embeddings)

print("✓ NearestNeighbors fitted")
print("Finding k-nearest neighbors for all nodes...")

# Find k-nearest neighbors
distances, indices = nbrs.kneighbors(embeddings)

print(f"✓ Found {config.K_NEIGHBORS} neighbors for each of {n_nodes:,} nodes")

# Build adjacency matrix from kNN results
print("\nBuilding sparse adjacency matrix...")
adjacency = sp.lil_matrix((n_nodes, n_nodes), dtype=np.float32)

for i in tqdm(range(n_nodes), desc="Building adjacency"):
    # Get neighbors (excluding self at index 0)
    neighbor_indices = indices[i, 1:]  # Skip first (self)
    neighbor_distances = distances[i, 1:]
    
    # Convert cosine distance to similarity: sim = 1 - dist
    neighbor_similarities = 1 - neighbor_distances
    
    if config.USE_BINARY_EDGES:
        # Binary edges (recommended for GCN stability)
        adjacency[i, neighbor_indices] = 1.0
    else:
        # Weighted edges (cosine similarity)
        adjacency[i, neighbor_indices] = neighbor_similarities

# Convert to CSR for efficiency
adjacency = adjacency.tocsr()

print(f"✓ Initial adjacency shape: {adjacency.shape}")
print(f"✓ Non-zero entries: {adjacency.nnz:,}")

# Make symmetric if requested
if config.MAKE_SYMMETRIC:
    print("\nMaking graph symmetric (undirected)...")
    # A_sym = (A + A^T) / 2 for weighted, or A | A^T for binary
    if config.USE_BINARY_EDGES:
        adjacency = (adjacency + adjacency.T).astype(bool).astype(float)
    else:
        adjacency = (adjacency + adjacency.T) / 2
    adjacency = sp.csr_matrix(adjacency)  # Convert back to CSR
    print(f"✓ Symmetric adjacency shape: {adjacency.shape}")
    print(f"✓ Non-zero entries: {adjacency.nnz:,}")

# Add self-loops (standard for GCN)
print("\nAdding self-loops...")
adjacency = adjacency + sp.eye(adjacency.shape[0], dtype=np.float32)
print(f"✓ With self-loops: {adjacency.nnz:,} edges")


STEP 3: BUILD kNN GRAPH (EFFICIENT METHOD)
Building graph with k=10 nearest neighbors...
Using sklearn NearestNeighbors (efficient for 122k documents)
✓ NearestNeighbors fitted
Finding k-nearest neighbors for all nodes...
✓ Found 10 neighbors for each of 122,817 nodes

Building sparse adjacency matrix...


Building adjacency: 100%|███████████████████████████████████████████████████| 122817/122817 [00:05<00:00, 22444.48it/s]


✓ Initial adjacency shape: (122817, 122817)
✓ Non-zero entries: 1,228,170

Making graph symmetric (undirected)...
✓ Symmetric adjacency shape: (122817, 122817)
✓ Non-zero entries: 2,086,542

Adding self-loops...
✓ With self-loops: 2,209,333 edges


In [11]:
# ============================================================================
# STEP 4: GRAPH STATISTICS & VALIDATION
# ============================================================================
print("\n" + "="*80)
print("STEP 4: GRAPH STATISTICS")
print("="*80)

# Convert to dense only for degree calculation (more efficient than working with sparse)
# Calculate degrees from sparse matrix
degrees = np.array(adjacency.sum(axis=1)).flatten()

# Basic stats
num_edges = adjacency.nnz
# For undirected graphs, each edge counted twice in nnz (except self-loops)
# Actual number of unique edges (excluding self-loops)
num_unique_edges = (num_edges - n_nodes) // 2 if config.MAKE_SYMMETRIC else (num_edges - n_nodes)
density = num_edges / (n_nodes * n_nodes)
avg_degree = degrees.mean()

print(f"Graph Statistics:")
print(f"  Nodes: {n_nodes:,}")
print(f"  Total entries (nnz): {num_edges:,}")
print(f"  Unique edges (excl. self-loops): {num_unique_edges:,}")
print(f"  Density: {density:.6f}")
print(f"  Average degree: {avg_degree:.2f}")
print(f"  Edge type: {'Binary (0/1)' if config.USE_BINARY_EDGES else 'Weighted (similarity)'}")
print(f"  Symmetric: {'Yes' if config.MAKE_SYMMETRIC else 'No'}")

# Degree distribution (already calculated above)
print(f"\nDegree Distribution:")
print(f"  Min degree: {degrees.min():.0f}")
print(f"  Max degree: {degrees.max():.0f}")
print(f"  Mean degree: {degrees.mean():.2f}")
print(f"  Median degree: {np.median(degrees):.2f}")

# Check for isolated nodes (degree = 0, excluding self-loops means degree = 1)
# A node is isolated if degree <= 1 (only self-loop)
isolated_nodes = np.sum(degrees <= 1)
if isolated_nodes > 0:
    print(f"⚠️  WARNING: {isolated_nodes} isolated nodes (only self-loop)")
else:
    print(f"✓ No isolated nodes (all have neighbors)")

# Check connectivity within splits
print(f"\nConnectivity within data splits:")
train_nodes = np.where(train_mask)[0]
val_nodes = np.where(val_mask)[0]
test_nodes = np.where(test_mask)[0]

# Use sparse matrix slicing for efficiency
train_subgraph = adjacency[train_nodes][:, train_nodes]
val_subgraph = adjacency[val_nodes][:, val_nodes]
test_subgraph = adjacency[test_nodes][:, test_nodes]

train_edges = train_subgraph.nnz
val_edges = val_subgraph.nnz
test_edges = test_subgraph.nnz

# Cross-split edges (sparse matrix indexing)
train_to_test = adjacency[train_nodes][:, test_nodes].nnz
train_to_val = adjacency[train_nodes][:, val_nodes].nnz
val_to_test = adjacency[val_nodes][:, test_nodes].nnz

print(f"  Train-Train edges: {train_edges:,}")
print(f"  Val-Val edges: {val_edges:,}")
print(f"  Test-Test edges: {test_edges:,}")
print(f"  Train-Test edges: {train_to_test:,} (enables semi-supervised learning!)")
print(f"  Train-Val edges: {train_to_val:,}")
print(f"  Val-Test edges: {val_to_test:,}")

# Check label homophily (similar nodes have same label?)
# ⚠️ WARNING: This uses test labels for analysis only - NOT for training!
print(f"\nLabel Homophily Analysis:")
print(f"⚠️  NOTE: Uses all labels (including test) for post-hoc analysis only")

# Get edges (use sparse matrix)
edge_indices = adjacency.nonzero()
# Filter out self-loops for homophily calculation
mask = edge_indices[0] != edge_indices[1]
edge_src = edge_indices[0][mask]
edge_dst = edge_indices[1][mask]

same_label_edges = np.sum(labels[edge_src] == labels[edge_dst])
total_edges_no_self = len(edge_src)
homophily = same_label_edges / total_edges_no_self if total_edges_no_self > 0 else 0

print(f"  Edges connecting same-label nodes: {same_label_edges:,} / {total_edges_no_self:,}")
print(f"  Homophily ratio: {homophily:.4f}")

if homophily > 0.5:
    print(f"  ✓ Good! High homophily suggests GCN will help")
elif homophily > 0.3:
    print(f"  ⚠️  Moderate homophily - GCN may still help")
else:
    print(f"  ⚠️  Low homophily - consider different graph construction")


STEP 4: GRAPH STATISTICS
Graph Statistics:
  Nodes: 122,817
  Total entries (nnz): 2,209,333
  Unique edges (excl. self-loops): 1,043,258
  Density: 0.000146
  Average degree: 17.99
  Edge type: Binary (0/1)
  Symmetric: Yes

Degree Distribution:
  Min degree: 11
  Max degree: 145
  Mean degree: 17.99
  Median degree: 16.00
✓ No isolated nodes (all have neighbors)

Connectivity within data splits:
  Train-Train edges: 34,129
  Val-Val edges: 11,161
  Test-Test edges: 1,613,141
  Train-Test edges: 177,234 (enables semi-supervised learning!)
  Train-Val edges: 10,329
  Val-Test edges: 87,888

Label Homophily Analysis:
⚠️  NOTE: Uses all labels (including test) for post-hoc analysis only
  Edges connecting same-label nodes: 1,783,602 / 2,086,516
  Homophily ratio: 0.8548
  ✓ Good! High homophily suggests GCN will help


In [12]:
# ============================================================================
# STEP 5: SAVE GRAPH
# ============================================================================
print("\n" + "="*80)
print("STEP 5: SAVE GRAPH")
print("="*80)

# adjacency is already sparse CSR from earlier steps
sp.save_npz(config.ADJ_OUTPUT_PATH, adjacency)
print(f"✓ Saved adjacency matrix to: {config.ADJ_OUTPUT_PATH}")
print(f"  Format: Sparse CSR")
print(f"  Size on disk: ~{adjacency.data.nbytes / 1e6:.2f} MB")

# Save edge list (optional, for inspection)
print(f"\nSaving edge list to: {config.EDGE_LIST_OUTPUT_PATH}")

# Get edges from sparse matrix
rows, cols = adjacency.nonzero()
edge_list = []
for i in range(min(1000, len(rows))):  # Only save first 1000
    r, c = rows[i], cols[i]
    if r <= c:  # Only upper triangle for undirected
        weight = adjacency[r, c]
        edge_list.append((r, c, weight))

with open(config.EDGE_LIST_OUTPUT_PATH, 'w') as f:
    f.write("# source target weight\n")
    for src, tgt, weight in edge_list:
        f.write(f"{src} {tgt} {weight:.4f}\n")
print(f"✓ Saved first {len(edge_list)} edges to: {config.EDGE_LIST_OUTPUT_PATH}")

# Save statistics
stats = {
    'graph_type': 'kNN (efficient NearestNeighbors)',
    'k_neighbors': config.K_NEIGHBORS,
    'edge_type': 'binary' if config.USE_BINARY_EDGES else 'weighted',
    'num_nodes': int(n_nodes),
    'total_entries': int(num_edges),
    'unique_edges': int(num_unique_edges),
    'density': float(density),
    'avg_degree': float(avg_degree),
    'is_symmetric': config.MAKE_SYMMETRIC,
    'includes_self_loops': True,  # Always added
    'degree_stats': {
        'min': float(degrees.min()),
        'max': float(degrees.max()),
        'mean': float(degrees.mean()),
        'median': float(np.median(degrees))
    },
    'split_stats': {
        'train_train_edges': int(train_edges),
        'val_val_edges': int(val_edges),
        'test_test_edges': int(test_edges),
        'train_test_edges': int(train_to_test),
        'train_val_edges': int(train_to_val),
        'val_test_edges': int(val_to_test)
    },
    'homophily': float(homophily),
    'isolated_nodes': int(isolated_nodes),
    'note': 'Homophily uses test labels for analysis only, not for training'
}

with open(config.GRAPH_STATS_PATH, 'w') as f:
    json.dump(stats, f, indent=2)
print(f"✓ Saved statistics to: {config.GRAPH_STATS_PATH}")



STEP 5: SAVE GRAPH
✓ Saved adjacency matrix to: adjacency_matrix.npz
  Format: Sparse CSR
  Size on disk: ~17.67 MB

Saving edge list to: edge_list.txt
✓ Saved first 1000 edges to: edge_list.txt
✓ Saved statistics to: graph_statistics.json


In [13]:
# ============================================================================
# FINAL SUMMARY
# ============================================================================
print("\n" + "="*80)
print("GRAPH BUILDING COMPLETE!")
print("="*80)

print("\n📁 Files created:")
print(f"  1. {config.ADJ_OUTPUT_PATH} - Sparse adjacency matrix")
print(f"  2. {config.EDGE_LIST_OUTPUT_PATH} - Edge list (first 1000)")
print(f"  3. {config.GRAPH_STATS_PATH} - Graph statistics")

print("\n📊 Graph Summary:")
print(f"  • {n_nodes:,} nodes (documents)")
print(f"  • {num_edges:,} edges (similarities)")
print(f"  • {avg_degree:.1f} average connections per document")
print(f"  • {homophily:.2%} of edges connect same-label documents")

print("\n✅ Ready for GCN training (Step 3)!")
print("="*80)


GRAPH BUILDING COMPLETE!

📁 Files created:
  1. adjacency_matrix.npz - Sparse adjacency matrix
  2. edge_list.txt - Edge list (first 1000)
  3. graph_statistics.json - Graph statistics

📊 Graph Summary:
  • 122,817 nodes (documents)
  • 2,209,333 edges (similarities)
  • 18.0 average connections per document
  • 85.48% of edges connect same-label documents

✅ Ready for GCN training (Step 3)!


In [14]:
# ============================================================================
# VERIFICATION CHECKLIST
# ============================================================================
print("\n" + "="*80)
print("VERIFICATION CHECKLIST")
print("="*80)

checks = [
    ("Embeddings loaded", True),
    ("Labels loaded", True),
    ("Masks loaded", True),
    ("No shape mismatches", True),
    ("No mask overlaps", True),
    ("All nodes masked", True),
    ("No NaN/Inf values", not (np.isnan(embeddings).any() or np.isinf(embeddings).any())),
    ("No isolated nodes", isolated_nodes == 0),
    ("Train-test edges exist", train_to_test > 0),
    ("Homophily > 30%", homophily > 0.3),
    ("Graph is symmetric", config.MAKE_SYMMETRIC),
]

all_passed = True
for check_name, passed in checks:
    status = "✅" if passed else "⚠️ "
    print(f"  {status} {check_name}")
    if not passed:

SyntaxError: incomplete input (4073292390.py, line 26)